In [1]:
import polars as pl
import os
import glob
import shutil
import json
from datetime import datetime, timedelta
from typing import Optional, Dict

In [2]:
today = datetime.now().strftime('%b_%d_%Y')

def input_data(folder_path: str, sheet_name: str = None) -> pl.DataFrame:
    """
    Reads Excel and CSV files from a folder and concatenates them.
    CSV files are read directly as Strings to bypass any inference errors.
    """
    file_paths = glob.glob(os.path.join(folder_path, "*.xlsx")) + \
                 glob.glob(os.path.join(folder_path, "*.csv"))
    
    df_list = []

    for file in file_paths:
        export_time = datetime.fromtimestamp(os.path.getmtime(file))
        file_name = os.path.basename(file)
        
        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name, engine='calamine')
            # Cast Excel columns to String to maintain consistency with CSV
            df = df.select(pl.all().cast(pl.String))
            
        elif file.endswith('.csv'):
            # THE FIX: Set infer_schema_length=0 to force Polars to read 
            # all columns as String natively. This prevents ComputeError 
            # when encountering mixed numeric types (e.g., integers followed by floats).
            df = pl.read_csv(
                file, 
                encoding="utf-8", 
                infer_schema_length=0,
                ignore_errors=True
            )
        
        df = df.with_columns([
            pl.lit(file_name).alias('File Name'),
            pl.lit(export_time).alias('Export Time')
        ])

        df_list.append(df)

    if not df_list:
        return pl.DataFrame()
    
    # Changed from 'vertical' to 'diagonal_relaxed' for better stability.
    # If one CSV has an extra column that another doesn't, this prevents a crash.
    return pl.concat(df_list, how='diagonal_relaxed')

def input_data_parquet(folder_path: str) -> pl.DataFrame:
    file_paths = glob.glob(os.path.join(folder_path, "*.parquet"))
    
    if not file_paths:
        return pl.DataFrame()

    lf_list = []

    for file in file_paths:
        export_time = datetime.fromtimestamp(os.path.getmtime(file))
        file_name = os.path.basename(file)

        lf = pl.scan_parquet(file).with_columns([
            pl.lit(file_name).alias('File Name'),
            pl.lit(export_time).alias('Export Time')
        ])
        lf_list.append(lf)

    return pl.concat(lf_list, how='vertical').collect()

def csv_to_parquet(input_folder: str, output_folder: str, schema_overrides: Optional[Dict] = None) -> None:
    os.makedirs(output_folder, exist_ok=True)
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

    if not csv_files:
        return
    if schema_overrides is None:
        schema_overrides = {
            "Promoter Score (Calc)": pl.Float64,
            "Detractor Score (Calc)": pl.Float64,
        }
    schema_overrides["Itinerary"] = pl.String

    for csv_file in csv_files:
        file_name = os.path.basename(csv_file)
        parquet_name = os.path.splitext(file_name)[0] + ".parquet"
        output_path = os.path.join(output_folder, parquet_name)

        try:
            pl.scan_csv(
                csv_file,
                infer_schema_length=10000,
                schema_overrides=schema_overrides,
                ignore_errors=True
            ).sink_parquet(output_path)
            
        except Exception as e:
            print(f"Critical failure while converting {file_name} to Parquet: {e}")


def copy_csv_files(source_folder: str, destination_folder: str) -> None:
    os.makedirs(destination_folder, exist_ok=True)
    files = glob.glob(os.path.join(source_folder, "*.csv"))
    for f in files: 
        shutil.copy2(f, destination_folder)

def restore_all_folders(backup_folder: str, paths_dict: dict, periods: list, mapping: dict) -> None:
    for key, identifier in mapping.items():
        if key not in paths_dict:
            continue
        dest_folder = paths_dict[key]
        os.makedirs(dest_folder, exist_ok=True)
        for period in periods:
            search_pattern = os.path.join(backup_folder, f"*{identifier}*{period}*.csv")
            matched_files = glob.glob(search_pattern)
            for file_path in matched_files:
                shutil.copy2(file_path, dest_folder)

def clean_raw_input_folders(folder_paths: dict, target_periods: list, file_patterns: dict):
    total_deleted = 0
    for folder_key, pattern in file_patterns.items():
        source_dir = folder_paths.get(folder_key)
        if not source_dir or not os.path.exists(source_dir):
            continue
        for period in target_periods:
            search_query = os.path.join(source_dir, f"*{pattern}*{period}*.*")
            files_to_delete = glob.glob(search_query)
            for filepath in files_to_delete:
                if os.path.exists(filepath):
                    try:
                        os.remove(filepath)
                        total_deleted += 1
                    except Exception as e: pass

def pre_process_fcr_excel(folder_path: str):
    excel_files = glob.glob(os.path.join(folder_path, "*.xlsx")) + glob.glob(os.path.join(folder_path, "*.xls"))
    
    for file_path in excel_files:
        try:
            df = pl.read_excel(file_path, engine="calamine", has_header=False)
            
            if df.height < 2:
                continue
            header_row_index = 0
            for i in range(min(10, df.height)):
                row_data = [str(x).strip() for x in df.row(i) if x is not None]
                if "FCR Category" in row_data or "Conversation Id" in row_data:
                    header_row_index = i
                    break
            raw_headers = df.row(header_row_index)
            headers = []
            seen = set()
            for i, c in enumerate(raw_headers):
                col_name = str(c).strip() if c is not None and str(c).strip() != "" else f"col_{i}"
                if col_name in seen:
                    col_name = f"{col_name}_dup_{i}"
                seen.add(col_name)
                headers.append(col_name)

            df = df.slice(header_row_index + 1)
            df.columns = headers
            csv_path = os.path.splitext(file_path)[0] + ".csv"
            df.write_csv(csv_path)
            # os.remove(file_path)
        except Exception:
            pass

In [3]:
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path = f"{first_glob}/Concentrix Corporation"
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Not found the path: {test_path}")

folder_paths = {
    "input_performance":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data',
    "output_performance_combine":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/OUTPUT_PERFORMANCE/OUTPUT_PERFORMANCE_COMBINE',
    "hc_extend_by_month":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month',
    "input_survey":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN',
    "input_t3":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN',
    "input_afcr":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR',
    "input_iex":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX',
    "mapping_file": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/AQG_summarized.xlsx',
    "global_hc": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/Global_HC.parquet',
    "input_parquet_performance":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/PARQUET',
    "input_csv_pulse":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Detractor_Compliance/0_source',
    "output_csv_pulse":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Detractor_Compliance/1_des',
    "input_xlsx_qa_audit":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Detractor_Compliance/2_qa_audit',
    "input_csv_re_direct":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/RE-DIRECT',
    "input_delayed_closure":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/DELAYED_CLOSURE',
    "csv":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/CSV',
}

print("--- FULL FOLDER PATHS LIST ---")
for key, path in folder_paths.items():
    print(f"{key}: {path}")

print("-" * 60)

pre_process_fcr_excel(folder_paths["input_afcr"])
csv_keys = ["input_performance", "input_t3", "input_csv_re_direct", "input_delayed_closure", "input_survey", "input_afcr"]
for k in csv_keys:
    src_folder = folder_paths[k]
    csv_to_parquet(src_folder, folder_paths["input_parquet_performance"])
    copy_csv_files(src_folder, folder_paths["csv"])

IEX = input_data(folder_paths["input_iex"])
IEX = IEX.unique()
IEX = IEX.with_columns([
    pl.col(['Date']).str.to_date("%Y-%m-%d", strict=False),
    pl.col(['Datetime_Fluctuate_Start_Shift','Datetime_Fluctuate_End_Shift','Datetime_First_Start_Shift','Datetime_First_End_Shift']).str.to_datetime("%Y-%m-%d %H:%M:%S.%f", strict=False),
    pl.col(['Night_Shift','Target','Unplanned','Planned','Roster Presented','Roster Scheduled']).cast(pl.Float64)])
columns_to_sec = ['Time_Of_Day','Open Time', 'Extra Time', 'Break Time', 'Lunch Time', 'Training', 'NCNS','AL','Target']
IEX = IEX.with_columns([(pl.col(col).fill_null(0).cast(pl.Float64) * 3600).alias(col) for col in columns_to_sec])
Night_Shift_1 = IEX[['Date','Email Id','Night_Shift']].unique()
Night_Shift_2 = Night_Shift_1.with_columns((pl.col('Date') - pl.duration(days=1)).alias('Previous Date'))
Night_Shift = Night_Shift_2.join(Night_Shift_1,left_on = ['Previous Date','Email Id'], right_on = ['Date','Email Id'], how = 'left')
Night_Shift = Night_Shift.rename({'Night_Shift_right':'Previous_Night_Shift'})
Previous_Date = IEX[['Date','Email Id','Datetime_First_Start_Shift','Shift Tracking']].unique()

--- FULL FOLDER PATHS LIST ---
input_performance: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data
output_performance_combine: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/OUTPUT_PERFORMANCE/OUTPUT_PERFORMANCE_COMBINE
hc_extend_by_month: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month
input_survey: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN
input_t3: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN
input_afcr: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR
input_iex: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX
mapping_file: C:/Users/huuchinh.nguyen/Concentrix Corporation/

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_44200\4097954441.py:42: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  pl.col(['Datetime_Fluctuate_Start_Shift','Datetime_Fluctuate_End_Shift','Datetime_First_Start_Shift','Datetime_First_End_Shift']).str.to_datetime("%Y-%m-%d %H:%M:%S.%f", strict=False),


In [4]:
target_periods = ["2026-01", "2026-02", "2026-03"]

file_patterns = {
    "input_performance": "New Look Excel data",
    "input_t3": "T3",
    "input_csv_re_direct": "re-direct",
    "input_delayed_closure": "Delayed Closure",
    "input_survey": "Survey Dump",
    "input_afcr": "FCR"
}

# restore_all_folders(folder_paths["csv"], folder_paths, target_periods, file_patterns)
# clean_raw_input_folders(folder_paths, target_periods, file_patterns)

In [5]:
SURVEY_INPUT = input_data(folder_paths["input_survey"])

survey_added_ir = SURVEY_INPUT.with_columns([
    pl.when(
        (pl.col("Question Category") == "Resolution") & (pl.col("Answer") == "Yes")
    ).then(1)
    .when(
        (pl.col("Question Category") == "Resolution") & (pl.col("Answer") == "No")
    ).then(0)
    .otherwise(0)
    .alias("_ir")
])

survey_ae = (
    SURVEY_INPUT
    .filter(pl.col("Question Category") == "agentExperience")
    .with_columns(
        pl.col("Answer")
          .cast(pl.Int64, strict=False)
          .alias("_ae")
    )
)

survey_verbatim = (
    SURVEY_INPUT
    .filter(pl.col("Question Category") == "Feedback")
    .with_columns([
        pl.col("Answer").str.replace_all(r"[\r\n\t]+", " ").alias("_verbatim"),
        pl.concat_str([
            pl.col("Agent Email ID").cast(pl.String).fill_null(""),
            pl.col("Conversation Id").cast(pl.String).fill_null(""),
            pl.col("Joined Time").str.to_datetime(strict=False).dt.strftime("%y%m%d%H%M%S")
        ], separator="_").alias("_conver_unique"),
    ])
)

metrics = ["d_happy_response", "d_surprised_response", "e_response", "t_response", "u_response"]

survey_duet = (
    SURVEY_INPUT
    .filter(pl.col("Question Category").is_in(metrics))
    .with_columns([
        pl.col("Question Category").alias("metric"),
        pl.col("Answer").cast(pl.Float64, strict=False).alias("value")
    ])
    .group_by(["Conversation Id", "metric"])
    .agg(pl.max("value").alias("value")) 
    .pivot(values="value", index="Conversation Id", columns="metric")
)

# --- Excel-equivalent calculations using column names (force Float64) ---
survey_duet = survey_duet.with_columns([
    pl.when(pl.any_horizontal([pl.col("d_happy_response").is_null(),
                               pl.col("d_surprised_response").is_null()]))
      .then(pl.lit(None, dtype=pl.Float64))
      .otherwise(pl.when((pl.col("d_happy_response") >= 4) & (pl.col("d_surprised_response") >= 3))
                   .then(100.0).otherwise(0.0))
      .alias("delight"),

    pl.when(pl.col("u_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("u_response") - 1.0) / 6.0) * 100.0).alias("usability"),

    pl.when(pl.col("e_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("e_response") - 1.0) / 6.0) * 100.0).alias("ease"),

    pl.when(pl.col("t_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("t_response") - 1.0) / 6.0) * 100.0).alias("trust"),

    pl.when(pl.any_horizontal([pl.col("d_happy_response").is_null(),
                               pl.col("d_surprised_response").is_null(),
                               pl.col("u_response").is_null(),
                               pl.col("e_response").is_null(),
                               pl.col("t_response").is_null()]))
      .then(pl.lit(None, dtype=pl.Float64))
      .otherwise(pl.when((pl.col("d_happy_response") >= 4) &
                         (pl.col("d_surprised_response") >= 3) &
                         ((pl.col("u_response") + pl.col("e_response")) >= 13) &
                         (pl.col("e_response") >= 6) &
                         (pl.col("t_response") >= 6))
                   .then(100.0).otherwise(0.0))
      .alias("DUET"),
])

# Round computed columns
survey_duet = survey_duet.with_columns([
    pl.col("delight").round(0),
    pl.col("usability").round(0),
    pl.col("ease").round(0),
    pl.col("trust").round(0),
    pl.col("DUET").round(0),
])

# Keep ALL required columns (raw + computed)
final_cols = ["Conversation Id"] + metrics + ["delight", "usability", "ease", "trust", "DUET"]
survey_duet = survey_duet.select([c for c in final_cols if c in survey_duet.columns])
survey_ir = (survey_added_ir.select(["Conversation Id", "_ir"]).filter(pl.col("_ir") == 1).unique())
survey_ae = (survey_ae.select(["Conversation Id", "_ae"]).filter(pl.col("_ae") > 0).unique())
verbatim = (survey_verbatim.select(["_conver_unique", "_verbatim"])).unique()

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_44200\1549389880.py:48: DeprecationWarning: The argument `columns` for `DataFrame.pivot` is deprecated. It has been renamed to `on`.
  .pivot(values="value", index="Conversation Id", columns="metric")


In [6]:
T3_INPUT = input_data(folder_paths["input_t3"])

t3_added_col_t3 = T3_INPUT.with_columns([
    pl.when(
        (pl.col("Agent Queue Group Name").str.contains("T3", literal=True)) &
        (pl.col("Transferred (Yes/No)") == "Yes")
    ).then(1).otherwise(0).alias("T3")])

t3_final = (t3_added_col_t3.select(["Conversation Id", "T3"]).filter(pl.col("T3") == 1).unique())

In [7]:
def process_afcr_folder(folder_path: str) -> pl.DataFrame:
    all_dataframes = []
    
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        timestamp = os.path.getmtime(file_path)
        export_time = datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d %H:%M:%S')
        
        df = pl.read_csv(
            file_path, 
            encoding="utf-8", 
            schema_overrides={"Itinerary": pl.String},
            infer_schema_length=10000,
            ignore_errors=True
        )
        
        cast_exprs = []
        for col, dtype in [("Handle Time", pl.Float64), ("Duet", pl.Float64), 
                           ("Passed Sessions", pl.Int64), ("Failed Sessions", pl.Int64)]:
            if col in df.columns:
                cast_exprs.append(pl.col(col).cast(dtype, strict=False))
                
        if cast_exprs:
            df = df.with_columns(cast_exprs)
        
        df = df.with_columns([
            pl.lit(file_name).alias('File Name'),
            pl.lit(export_time).alias('Export Time')
        ])
        all_dataframes.append(df)
        
    if not all_dataframes:
        return pl.DataFrame()
        
    return pl.concat(all_dataframes, how="diagonal_relaxed")


afcr_input = process_afcr_folder(folder_paths["input_afcr"])

if not afcr_input.is_empty():
    afcr_input = (
        afcr_input
        .filter(pl.col("Vendor Partner Location") == "Concentrix (Ho Chi Minh City)")
        .select([
            pl.col("Agent Email Address").alias("Agent Email ID"),
            pl.col("Conversation Id"),
            pl.col("Passed Sessions"),
            pl.col("Failed Sessions"),
            pl.when(pl.col("Passed Sessions").is_in([0, 1]))
            .then(1)
            .otherwise(0)
            .alias("Total Sessions")
        ])
        .unique()
    )

In [8]:
DELAYED_CLOSURE_INPUT = (
    input_data(folder_paths["input_delayed_closure"])
    .filter(pl.col("Agent Vendor Location") == "Concentrix (Ho Chi Minh City)")
    .unique(subset=["Agent Email ID", "Conversation Id", "Joined Time"], keep='last'))
delayed_closure = (
    DELAYED_CLOSURE_INPUT
    .select([
        "Agent Email ID", "Conversation Id", "Joined Time", "Traveler Unresponsive Time", "Response Time",
        "Agent Disconnect (Yes / No)", "Ghost (Yes / No)", "Requeued (Yes / No)"
    ])
    .with_columns([
        pl.col("Joined Time").str.to_datetime().cast(pl.Datetime),
        pl.col("Traveler Unresponsive Time").cast(pl.Float64),
        pl.col("Response Time").cast(pl.Float64),
        (pl.col("Traveler Unresponsive Time").cast(pl.Float64) - 240)
            .clip(0, None)
            .alias("Exceed Time"),
        ((pl.col("Traveler Unresponsive Time").cast(pl.Float64) - 240) > 0)
            .cast(pl.Int8)
            .alias("Exceed Chat"),
        (pl.col("Agent Disconnect (Yes / No)") == "Yes").cast(pl.Int8).alias("Agent Disconnect"),
        (pl.col("Ghost (Yes / No)") == "Yes").cast(pl.Int8).alias("Ghost"),
        (pl.col("Requeued (Yes / No)") == "Yes").cast(pl.Int8).alias("Requeued"),
        (pl.col("Response Time").cast(pl.Float64).is_null() | (pl.col("Response Time").cast(pl.Float64) == 0))
            .cast(pl.Int8)
            .alias("Traveler Unresponsive")
    ])
    .drop([
        "Agent Disconnect (Yes / No)", "Ghost (Yes / No)", "Requeued (Yes / No)",
        "Traveler Unresponsive Time", "Response Time"
    ])
    .group_by(["Agent Email ID", "Conversation Id", "Joined Time"])
    .agg([
        pl.sum("Exceed Time").alias("Exceed Time"),
        pl.sum("Exceed Chat").alias("Exceed Chat"),
        pl.sum("Agent Disconnect").alias("Agent Disconnect"),
        pl.sum("Ghost").alias("Ghost"),
        pl.sum("Requeued").alias("Requeued"),
        pl.sum("Traveler Unresponsive").alias("Traveler Unresponsive")
    ])
)

delayed_closure = delayed_closure.with_columns(
    pl.when(pl.col("Exceed Time") <= 0).then(pl.lit(None, dtype=pl.Utf8))
      .when((pl.col("Exceed Time")/60) <= 1).then(pl.lit("01 Mins"))
      .when((pl.col("Exceed Time")/60) <= 2).then(pl.lit("02 Mins"))
      .when((pl.col("Exceed Time")/60) <= 3).then(pl.lit("03 Mins"))
      .when((pl.col("Exceed Time")/60) <= 4).then(pl.lit("04 Mins"))
      .when((pl.col("Exceed Time")/60) <= 5).then(pl.lit("05 Mins"))
      .when((pl.col("Exceed Time")/60) <= 10).then(pl.lit("05-10 Mins"))
      .when((pl.col("Exceed Time")/60) <= 15).then(pl.lit("10-15 Mins"))
      .when((pl.col("Exceed Time")/60) <= 30).then(pl.lit("15-30 Mins"))
      .otherwise(pl.lit("30+ Min"))
      .alias("Exceed Bucket")
)
print(delayed_closure.height)

196302


In [9]:
# region PULSE
# def input_csv_pulse(folder_path: str) -> pl.DataFrame:
#     selected_columns = [
#         "AUDIT ID", "AUDITEE NAME", "AUDITEE EMPLOYEE ID", "AUDITEE SUPERVISOR NAME",
#         "AUDITEE SUPERVISOR EMPLOYEE ID", "AUDITOR NAME", "AUDITOR EMPLOYEE ID",
#         "DELEGEE NAME", "DELEGEE EMPLOYEE ID", "AUDIT DATE", "LOBS", "TRANSACTION ID",
#         "TRANSACTION DATE", "MONITORING TYPE", "ITINERARY NUMBER", "RESOLUTION AVAILABILITY",
#         "TRAVELER'S PAIN POINT", "ARTICLE NUMBER THAT SHOULD HAVE BEEN FOLLOWED?", "LOB #",
#         "PASTE THE SUB-TOPIC FROM ARTICLE (E.G. PROPERTY SAYS NO (INSIDE PENALTY))",
#         "LANGUAGE..", "TRIP STAGE.", "SUB - INTENT", "PRODUCT CATEGORY",
#         "GENERAL FEEDBACK ON THE AUDIT", "ACKNOWLEDGEMENT/DISPUTE DATE", "AUDIT STATUS",
#         "GENERAL COMMENT ON DISPUTE", "IS ESCALATED", "IS RESOLVED", "COACHING SCHEDULED",
#         "COACHING PUBLISHED DATE", "COACHING ACKNOWLEDGEMENT STATUS",
#         "AUDIT LAST MODIFIED DATE", "AUDIT MODIFIED BY", "ACTUAL AUDITOR"
#     ]

#     dfs = []

#     for filename in os.listdir(folder_path):
#         if filename.endswith(".csv"):
#             file_path = os.path.join(folder_path, filename)
#             print(f"📄 Đang đọc file: {file_path}")
#             df = pd.read_csv(file_path)
#             existing_cols = [col for col in selected_columns if col in df.columns]
#             df = df[existing_cols]
#             if "TRANSACTION ID" in df.columns:
#                 df["TRANSACTION ID"] = df["TRANSACTION ID"].astype(str).str.replace("'", "", regex=False)
#             if "AUDITEE EMPLOYEE ID" in df.columns and "TRANSACTION ID" in df.columns:
#                 df["OracleID_ConversationID_KEY"] = df["AUDITEE EMPLOYEE ID"].astype(str) + "_" + df["TRANSACTION ID"].astype(str)
#             dfs.append(df)

#     merged_df = pd.concat(dfs, ignore_index=True)
#     pl_df = pl.from_pandas(merged_df)

#     return pl_df

# PULSE_INPUT = input_csv_pulse(folder_paths["input_csv_pulse"])
# PULSE_INPUT
# endregion

In [10]:
#region QA AUDIT
# QA_AUDIT_INPUT = input_data(folder_paths["input_xlsx_qa_audit"])

# QA_AUDIT_INPUT = QA_AUDIT_INPUT.with_columns([
#     pl.concat_str([
#         pl.col("Agent Email").cast(str),
#         pl.col("Conversation ID").cast(str),
#     ], separator="_")
#     .alias("EmailID_ConversationID_KEY"),
#     pl.lit(1).alias("QA_Audit")
# ])
# QA_AUDIT_INPUT_COLS = ["EmailID_ConversationID_KEY", "Case #", "AutoFail", "Evaluator Name", "Evaluation Date" ,"QA_Audit"]
# QA_AUDIT_INPUT = QA_AUDIT_INPUT.select(QA_AUDIT_INPUT_COLS)
# QA_AUDIT_INPUT
#endregion

In [11]:
RE_DIRECT_INPUT = (
    input_data(folder_paths["input_csv_re_direct"])
    .with_columns(
        PeopleID_ConversationID_KEY=pl.concat_str(
            [pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_",
        ),
        Re_Direct=pl.lit(1),
        **{
            "Re-Direct Text": (
                pl.col("Text").cast(pl.Utf8).fill_null("")
                .str.replace_all(r"(\r\n|\r|\n)+", " | ")
                .str.replace_all(r"[\-•\u2022\u25CF\u25E6\u2043\u2219\u00B7\u2013\u2014]+", "")
                .str.replace_all(r"\s{2,}", " ")
                .str.strip_chars()
            )
        }
    )
    .select(["PeopleID_ConversationID_KEY", "Re_Direct", "Re-Direct Text"])
    .unique(maintain_order=True)
)

In [12]:
PERFORMANCE_INPUT = input_data(folder_paths["input_performance"])

try: 
    if PERFORMANCE_INPUT.columns[0] == "":
        PERFORMANCE_INPUT = PERFORMANCE_INPUT.drop(PERFORMANCE_INPUT.columns[0])
except: pass

print(PERFORMANCE_INPUT.columns)
existing_cols = set(PERFORMANCE_INPUT.columns)

columns_to_cast = {
    "Joined Time": pl.Datetime,
    "Handle Time (Sum)": pl.Float64,
    "Handle (Count)": pl.Int64,
    "Hold Time (Sum)": pl.Float64,
    "Talk Time (Sum)": pl.Float64,
    "Wrap Up Time (Sum)": pl.Float64,
    "Survey Submitted (Count)":pl.Float64,
    "Promoter Score (Calc)": pl.Float64,
    "Detractor Score (Calc)": pl.Float64,
    "Neutral Score (Calc)": pl.Float64,
    "Has Followup Time": pl.Float64,
    "Response Time (Sum)": pl.Float64,
    "Response Time (Avg)": pl.Float64,
    "Concurrency": pl.Float64,
    "Survey Offered (Count)":pl.Int64,
}

datetime_cols = ["Started Time", "Joined Time", "Submitted Time", "Left Time"]

casts = []

for col, dtype in columns_to_cast.items():
    if col in existing_cols:
        if col == "Submitted Date":
            casts.append(pl.col(col).str.strptime(pl.Date, "%m/%d/%Y", strict=False))
        elif col in datetime_cols:
            casts.append(
                pl.when(pl.col(col).str.contains(r"^\d{4}-\d{2}-\d{2}"))  # ISO: YYYY-MM-DD
                  .then(pl.col(col).str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False))
                .when(pl.col(col).str.contains(r"^\d{1,2}/\d{1,2}/\d{4}"))  # US: M/D/YYYY
                  .then(pl.col(col).str.strptime(pl.Datetime, "%m/%d/%Y %H:%M", strict=False))
                .otherwise(None)
                .alias(col)
            )
        elif dtype == pl.Float64:
            casts.append(
                pl.col(col)
                .cast(pl.Utf8)
                .str.replace_all(",", "")
                .cast(pl.Float64, strict=False)
                .alias(col)
            )
        else:
            casts.append(pl.col(col).cast(dtype).alias(col))

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_INPUT.with_columns(casts)
PERFORMANCE_ADDED_CCR72 = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.when(pl.col("Has Followup Time").cast(pl.Float64) <= 4320)
      .then(1.0)
      .otherwise(0.0)
      .alias("CCR72")
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_ADDED_CCR72.with_columns([
    pl.col("Joined Time").dt.date().alias("Joined Date")
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_NEXT_STEP.with_columns([
    (pl.col("Joined Time") + pl.duration(hours=14)).alias("Join Time (VNT)"),
    (pl.col("Joined Time") + pl.duration(hours=14)).dt.date().alias("Join Date (VNT)")
])

HC_MASTER_DATABASE = input_data(folder_paths["hc_extend_by_month"])
HC_MASTER_DATABASE = HC_MASTER_DATABASE.rename({'Date Start Week': 'Week_Monday'})

HC_MASTER_DATABASE = HC_MASTER_DATABASE.with_columns([
    pl.col('Date').str.strptime(pl.Date, "%Y-%m-%d", strict=False)
])

hc_master_selected = HC_MASTER_DATABASE.select([
    "Date","Email Id", "OracleID", "People ID", "IEX ID", "Employee Name", "Alias", "Designation", "Detail Status", "Active",
    "Supervisor Name", "Wave", "LOB", 'LG Tenure', 'NL Tenure', 'Mini TL - Email', 'Mini TL - Short Name', 'Mini TL Start Date', 'Site'
]).unique()

hc_master_selected = hc_master_selected.rename({'Mini TL - Short Name': 'Mini TL', 'LOB':'Group'})
performance_merged = PERFORMANCE_NEXT_STEP.join(
    hc_master_selected,
    left_on=["Joined Date","Agent Email ID"],
    right_on=["Date","Email Id"],
    how="left")
GLOBAL_HC = pl.read_parquet(folder_paths["global_hc"])
global_hc_clean = GLOBAL_HC.select(["SSO ID","Production Start date","Agent/Non Agent"]).unique(subset=["SSO ID"], keep="first")
merged_global_hc = performance_merged.join(global_hc_clean,left_on="Agent Email ID",right_on="SSO ID",how="left")
merged_iex = merged_global_hc.join(IEX[['Date','Email Id','First Shift','Datetime_First_Start_Shift','Night_Shift']], left_on=['Join Date (VNT)','Agent Email ID'], right_on=['Date','Email Id'], how='left')
mapping = pl.read_excel(folder_paths["mapping_file"])
performance_cleaned = merged_iex.join(mapping, on="Agent Queue Group Name",how='left')
lc_mapping = pl.read_excel(folder_paths["mapping_file"], sheet_name="kpi")

['Joined Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Queue Group Name', 'Latest VA Intent', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Latest VA Product', 'Response Count', 'Response Time', 'Customer Loyalty Status', 'Business Segment Name', 'Partner Name', 'Language', 'Effective Disconnected By', 'Joined Time', 'Left Time', 'Entry Point App', 'Device Category', 'Device Os', 'Browser', 'Has Followup Time', 'Queue Name', 'Agent Manager Name', 'Agent Name', 'Handle (Count)', 'Handle Time (Sum)', 'Wrap Up Time (Sum)', 'Hold Time (Sum)', 'Talk Time (Sum)', 'Follow Up (Count)', 'Survey Submitted (Count)', 'Promoter Score (Calc)', 'Detractor Score (Calc)', 'Assigned Agent Time (Sum)', 'Sum of Chat Agent First Response Time', 'Survey Offered (Count)', 'File Name', 'Export Time']


In [13]:
performance_updated_ns = performance_cleaned.with_columns((pl.col('Join Date (VNT)') - pl.duration(days=1)).alias('Previous Date'))
performance_updated_ns = performance_updated_ns.join(Night_Shift[['Date','Email Id','Night_Shift','Previous_Night_Shift']],
                                                                 left_on=['Join Date (VNT)','Agent Email ID'],
                                                                 right_on=['Date','Email Id'],
                                                                 how='left')
def update_night_shift(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.when(
            (pl.col('Night_Shift') == 0) & 
            (pl.col('Join Time (VNT)').dt.time() < pl.time(17, 0)) & 
            (pl.col('Join Time (VNT)').dt.time() >= pl.time(0, 0)) & 
            (pl.col('Previous_Night_Shift') == 1)
        ).then(False).otherwise(True).alias('Night_Shift_2_Check')
    )

    df = df.with_columns(
        pl.when(pl.col('Night_Shift_2_Check') == False)
        .then(1)
        .otherwise(pl.col('Night_Shift'))
        .alias('Night_Shift')
    )

    df = df.with_columns(
        pl.when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 12) & (pl.col('Previous_Night_Shift') == 1))
        .then(pl.col('Join Date (VNT)') - pl.duration(days=1))
        
        .when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 12) & (pl.col('Night_Shift') == 1))
        .then(pl.col('Join Date (VNT)') - pl.duration(days=1))
        
        .when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 18) & (pl.col('Night_Shift') == 0))
        .then(pl.col('Join Date (VNT)'))
        
        .when((pl.col('Join Time (VNT)').dt.hour() >= 18) & (pl.col('Night_Shift') == 1))
        .then(pl.col('Join Date (VNT)'))
        
        .otherwise(pl.col('Join Date (VNT)'))
        .alias('_Date_Converted')
    )

    return df

performance_updated_ns = update_night_shift(performance_updated_ns)

In [14]:
performance_processed = performance_updated_ns.with_columns([
    (1 - pl.col("Promoter Score (Calc)") - pl.col("Detractor Score (Calc)"))
        .alias("Neutral Score (Calc)")
])

performance_processed = performance_processed.with_columns([
    (pl.col("Promoter Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_promoter"),
    (pl.col("Detractor Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_detractor"),
    (pl.col("Neutral Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_neutral")
])

performance_processed = performance_processed.with_columns([
    pl.when(pl.col("Initiated Outbound (Yes / No)") == "Yes").then(1).otherwise(0).alias("_aob"),

    pl.col("Survey Submitted (Count)").alias("_survey"),
    pl.col("CCR72").cast(pl.Float64).fill_null(0.0).alias("_fup_72"),
    (1.0 - pl.col("CCR72").cast(pl.Float64).fill_null(0.0)).alias("_rr"),

    pl.col("Joined Time").dt.date().alias("_PST.Date"),
    pl.col("Joined Time").dt.strftime("%y_%m").alias("_PST.Month"),
    pl.col("Joined Time").dt.strftime("%G_%V").str.slice(2).alias("_PST.Week"),
    pl.col("Joined Time").dt.year().alias("_PST.Year"),

    pl.concat_str([
        pl.col("Agent Email ID").cast(pl.Utf8).fill_null(""),
        pl.col("Conversation Id").cast(pl.Utf8).fill_null(""),
        pl.col("Joined Time").dt.strftime("%y%m%d%H%M%S")
    ], separator="_").alias("_conver_unique"),

    pl.when(pl.col("Group").is_in(["Non_Lodging", "Lodging"])).then(pl.lit("agent")).otherwise(None).alias("Agent"),
    pl.col("_Date_Converted").alias("_Date"),
    pl.col("Survey Offered (Count)").alias("_offer"),
    pl.when(pl.col("Group") == "Lodging").then(pl.lit("LG Tenure"))
     .when(pl.col("Group") == "Non_Lodging").then(pl.lit("NL Tenure"))
     .otherwise(None)
     .alias("Tenure")
])
performance_processed = performance_processed.with_columns(
    (
        pl.col("_PST.Date").cast(pl.Date) -
        pl.col("Production Start date")
          .cast(pl.String)
          .str.to_date("%Y-%m-%d %H:%M:%S", strict=False)
          .cast(pl.Date)
    )
    .dt.total_days()
    .cast(pl.Int32)
    .alias("AON_Days")
)
performance_processed = performance_processed.with_columns([
    pl.when(
        pl.col("AON_Days").is_null() & (
            (pl.col("Agent/Non Agent") == "Agent") | (pl.col("Agent/Non Agent") == "ID Deleted")
        )
    )
    .then(pl.lit("Nesting"))
    .when(pl.col("AON_Days") > 180)
      .then(pl.lit("> 180 Days"))
    .when(pl.col("AON_Days") >= 91)
      .then(pl.lit("91 - 180"))
    .when(pl.col("AON_Days") >= 61)
      .then(pl.lit("61 - 90"))
    .when(pl.col("AON_Days") >= 31)
      .then(pl.lit("31 - 60"))
    .when(pl.col("AON_Days") >= 0)
      .then(pl.lit("00 - 30"))
    .otherwise(None)
    .alias("AON Status")
])
performance_processed = performance_processed.join_asof(
    lc_mapping,
    left_on="_PST.Date",
    right_on="Effective Date",
    by="LOB",
    strategy="backward"
)
performance_processed = performance_processed.with_columns([
    (pl.col("Handle Time (Sum)") >= pl.col("Threshole_LC")).cast(pl.Int8).alias("_lc")
])

performance_processed = performance_processed.with_columns([
    (pl.col("Handle Time (Sum)") < 240).cast(pl.Int8).alias("Short Chat")
])

performance_processed = performance_processed.with_columns([
    pl.concat_str([pl.col("Agent Email ID"), pl.col("_PST.Date").dt.strftime("%y%m%d")]).alias("KEY")
])

performance_processed = performance_processed.with_columns([
    pl.concat_str([pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],separator="_").alias("EmailID_ConversationID_KEY"),
    pl.concat_str([pl.col("OracleID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],separator="_").alias("OracleID_ConversationID_KEY"),
    pl.concat_str([pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],separator="_").alias("PeopleID_ConversationID_KEY"),
])

performance_processed = performance_processed.with_columns([
    (
        pl.when(pl.col("_promoter") > 0)
          .then(pl.lit("promoter, "))
          .otherwise(pl.lit(""))
      +
        pl.when(pl.col("_detractor") > 0)
          .then(pl.lit("detractor, "))
          .otherwise(pl.lit(""))
      +
        pl.when(pl.col("_neutral") > 0)
          .then(pl.lit("neutral"))
          .otherwise(pl.lit(""))
    ).str.strip_chars(", ").alias("_nps_type")
])

def get_30min_interval(dt):
    if dt is None:
        return None
    hour = dt.hour
    minute = dt.minute
    if minute < 30:
        start = datetime(dt.year, dt.month, dt.day, hour, 0)
        end = start + timedelta(minutes=29)
    else:
        start = datetime(dt.year, dt.month, dt.day, hour, 30)
        end = start + timedelta(minutes=29)
    return f"{start.strftime('%H:%M')}-{end.strftime('%H:%M')}"

def get_period(dt):
    if dt is None:
        return None
    hour = dt.hour
    if hour >= 18:
        return 'Night'
    elif hour >= 12:
        return 'Mid'
    else:
        return 'Morning'

performance_processed = performance_processed.with_columns([
    pl.col('Joined Time').map_elements(get_30min_interval, return_dtype=pl.Utf8).alias('_PST.Interval'),
    pl.col('Join Time (VNT)').map_elements(get_30min_interval, return_dtype=pl.Utf8).alias('_VNT.Interval'),
    pl.col('Datetime_First_Start_Shift').map_elements(get_period, return_dtype=pl.Utf8).alias('_VNT.Period')
])

performance_merged_survey_t3 = (
    performance_processed
    .join(survey_ir, on = "Conversation Id", how="left")
    .join(survey_ae, on = "Conversation Id", how="left")
    .join(survey_duet, on = "Conversation Id", how="left")
    .join(t3_final, on = "Conversation Id", how="left")
    .join(delayed_closure, on = ["Agent Email ID", "Conversation Id", "Joined Time"], how="left")
    .join(RE_DIRECT_INPUT, on = "PeopleID_ConversationID_KEY", how="left")
    .join(verbatim, on = ["_conver_unique"], how="left")
    .join(afcr_input, on = ["Agent Email ID", "Conversation Id"], how="left")
)

print(performance_merged_survey_t3.columns)
print(performance_merged_survey_t3.select(["_PST.Date", "Production Start date", "AON_Days"]).head())

selected_columns = [
    "Export Time","File Name", "Agent People Id", "Business Segment Name", "Partner Name", "Customer Loyalty Status", 
    "Response Count", "Response Time", "Latest VA Product", "Language", "Latest VA Intent", "Conversation Id",
    "Agent Queue Group Name","Joined Time","_PST.Interval", "Agent Email ID","Left Time", "Handle (Count)", "Handle Time (Sum)", "Hold Time (Sum)", "Talk Time (Sum)",
    "Join Time (VNT)","_VNT.Interval","_VNT.Period","Wrap Up Time (Sum)", "Agent Business Location",
    "CCR72", "_PST.Date", "_PST.Month", "_PST.Year", "_aob", "LOB", "_conver_unique", "_nps_type", "_promoter", "_detractor", "_neutral", 
    "_survey", "_ir", "_ae", "d_happy_response", "d_surprised_response", "e_response", "t_response", "u_response", 'delight', 'usability', 'ease', 'trust', 'DUET', 
    "_offer", "T3", "Re_Direct", "Re-Direct Text", "Exceed Time", "Exceed Chat", "Agent Disconnect", "Ghost", "Requeued", "_verbatim",
    "Exceed Bucket", "_PST.Week", "_lc", "AON Status", "Agent/Non Agent", "Tenure", "OracleID", "People ID", 
    "IEX ID", "Employee Name", "Alias", "Designation", "Detail Status", "Active", "Supervisor Name",
    "Wave", "Group", "_Date", "Mini TL - Email", "Mini TL", "Mini TL Start Date","Site", "Traveler Unresponsive", "Short Chat",  "Passed Sessions", "Failed Sessions", "Total Sessions"
]

fcr_columns = [
    "LOB","OracleID_ConversationID_KEY","EmailID_ConversationID_KEY","_nps_type","_detractor","_survey","_PST.Week","AON Status",
    "Agent/Non Agent","Tenure","Employee Name","Alias","Designation","Detail Status","Supervisor Name",
    "Wave","Group","_Date","Mini TL - Email","Mini TL","Mini TL Start Date","Site","Agent Business Location"
]

missing_cols = [col for col in selected_columns if col not in performance_merged_survey_t3.columns]
print("Các cột bị thiếu:", missing_cols)

performance_filtered = performance_merged_survey_t3.select(selected_columns)
# fcr_filtered = performance_merged_survey_t3.filter((pl.col("_detractor") > 0) & pl.col("Agent Business Location").str.contains("Ho Chi Minh", literal=True)).select(fcr_columns)

# fcr_final = (
#     fcr_filtered
#     .join(PULSE_INPUT, on = "OracleID_ConversationID_KEY", how = "left")
#     .join(QA_AUDIT_INPUT, on = "EmailID_ConversationID_KEY", how = "left"))

# fcr_final.write_csv(f'{folder_paths["output_csv_pulse"]}/fcr.csv')
performance_filtered = performance_filtered.sort(
    by=["Conversation Id", "Agent Email ID", "Joined Time"],
    descending=[False, False, True]
).with_columns(
    (pl.col("Joined Time").cum_count().over(["Conversation Id", "Agent Email ID"]) > 1)
    .cast(pl.Int8)
    .alias("Duplicate_Flag")
)
performance_filtered = performance_filtered.unique()

['Joined Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Queue Group Name', 'Latest VA Intent', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Latest VA Product', 'Response Count', 'Response Time', 'Customer Loyalty Status', 'Business Segment Name', 'Partner Name', 'Language', 'Effective Disconnected By', 'Joined Time', 'Left Time', 'Entry Point App', 'Device Category', 'Device Os', 'Browser', 'Has Followup Time', 'Queue Name', 'Agent Manager Name', 'Agent Name', 'Handle (Count)', 'Handle Time (Sum)', 'Wrap Up Time (Sum)', 'Hold Time (Sum)', 'Talk Time (Sum)', 'Follow Up (Count)', 'Survey Submitted (Count)', 'Promoter Score (Calc)', 'Detractor Score (Calc)', 'Assigned Agent Time (Sum)', 'Sum of Chat Agent First Response Time', 'Survey Offered (Count)', 'File Name', 'Export Time', 'CCR72', 'Join Time (VNT)', 'Join Date (VNT)', 'OracleID', 'People ID', 'IEX ID', 'Employee Name', 'Alias', 'Designation', 'Detail Status', 'Active', 'Supervisor Name', 'Wav

In [15]:
performance_filtered

Export Time,File Name,Agent People Id,Business Segment Name,Partner Name,Customer Loyalty Status,Response Count,Response Time,Latest VA Product,Language,Latest VA Intent,Conversation Id,Agent Queue Group Name,Joined Time,_PST.Interval,Agent Email ID,Left Time,Handle (Count),Handle Time (Sum),Hold Time (Sum),Talk Time (Sum),Join Time (VNT),_VNT.Interval,_VNT.Period,Wrap Up Time (Sum),Agent Business Location,CCR72,_PST.Date,_PST.Month,_PST.Year,_aob,LOB,_conver_unique,_nps_type,_promoter,_detractor,_neutral,…,T3,Re_Direct,Re-Direct Text,Exceed Time,Exceed Chat,Agent Disconnect,Ghost,Requeued,_verbatim,Exceed Bucket,_PST.Week,_lc,AON Status,Agent/Non Agent,Tenure,OracleID,People ID,IEX ID,Employee Name,Alias,Designation,Detail Status,Active,Supervisor Name,Wave,Group,_Date,Mini TL - Email,Mini TL,Mini TL Start Date,Site,Traveler Unresponsive,Short Chat,Passed Sessions,Failed Sessions,Total Sessions,Duplicate_Flag
datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,datetime[μs],str,str,str,i64,f64,f64,f64,datetime[μs],str,str,f64,str,f64,date,str,i32,i32,str,str,str,f64,f64,f64,…,i32,i32,str,f64,i64,i64,i64,i64,str,str,str,i8,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,str,str,str,str,i64,i8,i64,i64,i32,i8
2026-03-05 12:52:56,"""New Look Excel data_EN 2026-02…","""330139460""","""Travelocity United States""","""Travelocity""",null,"""0""",null,"""PACKAGE""","""en_US""","""VIEW_BOOKING""","""dc995131-8513-4741-992c-e4d020…","""Chat_Non_EN_Translation""",2026-02-25 11:35:36,"""11:30-11:59""","""shilpika.ghosh@concentrix.com""","""2026-02-25 11:35:44""",1,8.463,0.0,0.0,2026-02-26 01:35:36,"""01:30-01:59""",null,0.0,"""Concentrix (Kolkata)""",0.0,2026-02-25,"""26_02""",2026,0,"""LIO Chat""","""shilpika.ghosh@concentrix.com_…","""""",null,null,null,…,null,null,null,null,null,null,null,null,null,null,"""26_09""",0,"""Nesting""","""Agent""",null,null,null,null,null,null,null,null,null,null,null,null,2026-02-26,null,null,null,null,null,1,null,null,null,0
2026-04-04 10:55:03,"""New Look Excel data_EN 2026-03…","""411722561""","""Expedia United States""","""Expedia""",null,"""0""",null,"""UNKNOWN""","""en_US""","""UNKNOWN""","""efbbf44f-e0e8-4228-97c8-42cdb7…","""Voice_OD_GLB_EN_Lodging_Profic…",2026-03-29 11:31:48,"""11:30-11:59""","""surya.raut@concentrix.com""","""2026-03-29 11:48:48""",1,1020.382,544.393,459.99702,2026-03-30 01:31:48,"""01:30-01:59""",null,15.992,"""Concentrix (Kolkata)""",0.0,2026-03-29,"""26_03""",2026,1,"""LG Voice""","""surya.raut@concentrix.com_efbb…","""""",null,null,null,…,null,null,null,null,null,null,null,null,null,null,"""26_13""",0,"""Nesting""","""Agent""",null,null,null,null,null,null,null,null,null,null,null,null,2026-03-30,null,null,null,null,null,0,null,null,null,0
2026-05-02 13:37:16,"""New Look Excel data_EN 2026-04…","""56817464""","""Expedia Japan""","""Expedia""",null,"""0""",null,"""UNKNOWN""","""en_GB""","""UNKNOWN""","""4528bb75-7726-4fdc-8186-ec4b04…","""Voice_OD_Expert_GLB_EN""",2026-04-12 16:02:17,"""16:00-16:29""","""omkar.dimble@concentrix.com""","""2026-04-12 16:32:06""",1,1789.32,0.0,1789.32,2026-04-13 06:02:17,"""06:00-06:29""",null,0.0,"""Concentrix (Pune)""",1.0,2026-04-12,"""26_04""",2026,0,"""NL Voice""","""omkar.dimble@concentrix.com_45…","""""",null,null,null,…,null,null,null,null,null,null,null,null,null,null,"""26_15""",0,"""Nesting""","""Agent""",null,null,null,null,null,null,null,null,null,null,null,null,2026-04-13,null,null,null,null,null,0,null,null,null,1
2026-04-04 10:55:03,"""New Look Excel data_EN 2026-03…","""862367902""","""Expedia United States""","""Expedia""","""Blue""","""6""","""295.2300109863281""","""CAR""","""en_US""","""AMENITIES""","""5ea05c60-d5e1-47d0-8061-b96131…","""Chat_OD_EN_Car_Activity""",2026-03-02 08:53:39,"""08:30-08:59""","""minhkimcuong.trinh@concentrix.…","""2026-03-02 09:02:17""",1,519.0,0.0,0.0,2026-03-02 22:53:39,"""22:30-22:59""","""Night""",13.379,"""Concentrix (Ho Chi Minh City)""",0.0,2026-03-02,"""26_03""",2026,0,

In [16]:
# 1) Read the removal list from Excel
removed_id = pl.read_excel(folder_paths["mapping_file"], sheet_name="id_removed")

# 2) Helper to normalize Conversation Id (cast -> drop quotes -> trim spaces)
def clean_id(expr: pl.Expr) -> pl.Expr:
    return (
        expr.cast(pl.Utf8)
            .str.replace_all(r'["“”]', "")      # remove any quote characters
            .str.replace_all(r'^\s+|\s+$', "")  # trim leading/trailing whitespace (regex)
    )

# Validate required column
if "Conversation Id" not in removed_id.columns:
    raise ValueError("Sheet 'id_removed' must contain a 'Conversation Id' column.")

# Canonical removal list: unique, non-null, non-empty
removed_id_norm = (
    removed_id
    .select(clean_id(pl.col("Conversation Id")).alias("Conversation Id"))
    .drop_nulls()
    .filter(pl.col("Conversation Id") != "")
    .unique()
)

# 3) Instead of removing rows, add a flag column "IDs Removed" (Yes/No)
performance_filtered = (
    performance_filtered
    .with_columns(clean_id(pl.col("Conversation Id")).alias("__conv_norm"))
    .join(
        removed_id_norm
            .select(pl.col("Conversation Id").alias("__conv_norm"))
            .with_columns(pl.lit(1).alias("__rm")),
        on="__conv_norm",
        how="left",
    )
    .with_columns(pl.col("__rm").fill_null(0).alias("IDs Removed"))
    .drop(["__conv_norm", "__rm"])
)

# (Optional) thống kê nhanh
flagged = performance_filtered.select(pl.col("IDs Removed").sum().alias("flagged")).item()
total = performance_filtered.height
print(f"Flagged (IDs Removed=1): {flagged} / {total} rows")

Flagged (IDs Removed=1): 808 / 1419681 rows


In [17]:
performance_unique = performance_filtered.drop(["Export Time", "File Name"])
performance_unique = performance_unique.unique()

performance_hcm = performance_unique.filter(pl.col("Agent Business Location").str.contains("Ho Chi Minh", literal=True))
performance_all_site = performance_unique

In [18]:
print(performance_hcm.columns)

['Agent People Id', 'Business Segment Name', 'Partner Name', 'Customer Loyalty Status', 'Response Count', 'Response Time', 'Latest VA Product', 'Language', 'Latest VA Intent', 'Conversation Id', 'Agent Queue Group Name', 'Joined Time', '_PST.Interval', 'Agent Email ID', 'Left Time', 'Handle (Count)', 'Handle Time (Sum)', 'Hold Time (Sum)', 'Talk Time (Sum)', 'Join Time (VNT)', '_VNT.Interval', '_VNT.Period', 'Wrap Up Time (Sum)', 'Agent Business Location', 'CCR72', '_PST.Date', '_PST.Month', '_PST.Year', '_aob', 'LOB', '_conver_unique', '_nps_type', '_promoter', '_detractor', '_neutral', '_survey', '_ir', '_ae', 'd_happy_response', 'd_surprised_response', 'e_response', 't_response', 'u_response', 'delight', 'usability', 'ease', 'trust', 'DUET', '_offer', 'T3', 'Re_Direct', 'Re-Direct Text', 'Exceed Time', 'Exceed Chat', 'Agent Disconnect', 'Ghost', 'Requeued', '_verbatim', 'Exceed Bucket', '_PST.Week', '_lc', 'AON Status', 'Agent/Non Agent', 'Tenure', 'OracleID', 'People ID', 'IEX ID',

In [19]:
# request_file_path = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/conversation_id.xlsx'
# df_request = pl.read_excel(request_file_path, engine='calamine')

# df_request = df_request.with_columns([
#     pl.col("Conversation Id").cast(pl.Utf8).str.strip_chars()
# ])

# perf_source = performance_all_site.with_columns([
#     ((pl.col("Handle Time (Sum)") + pl.col("Hold Time (Sum)") + pl.col("Wrap Up Time (Sum)")) / 
#      pl.col("Handle (Count)")).alias("AHT_Calc")
# ])

# result_df = df_request.join(
#     perf_source,
#     on="Conversation Id",
#     how="left"
# )

# final_output = result_df.select([
#     "Agent Email ID",
#     "Conversation Id",
    
    
#     pl.col("_promoter").alias("Promoter"),
#     pl.col("_detractor").alias("Detractor"),
#     pl.col("_survey").alias("Survey"),
#     "DUET",
#     pl.col("AHT_Calc").alias("AHT")
# ])

# print(f"Extracted {final_output.height} rows.")
# print(final_output.head())

# final_output.write_excel('extracted_metrics_report.xlsx')

In [20]:
# region EXPORT
output_dir = folder_paths["output_performance_combine"]

for (month_value,), group in performance_hcm.group_by(['_PST.Month'], maintain_order=True):
    base_name = str(month_value)
    group.write_csv(os.path.join(output_dir, f"{base_name}.csv"))
    group.write_parquet(os.path.join(output_dir, f"{base_name}_performance_en_vn.parquet"))

parquet_files = glob.glob(os.path.join(output_dir, "*_performance_en_vn.parquet"))
out_path_total = os.path.join(output_dir, "_performance_hcm.parquet")

lazy_frames = [pl.scan_parquet(f) for f in parquet_files]

(
    pl.concat(lazy_frames, how="diagonal_relaxed") 
    .collect()
    .write_parquet(out_path_total)
)
# endregion

In [21]:
print(performance_all_site['_PST.Month'].unique())

shape: (5,)
Series: '_PST.Month' [str]
[
	"26_01"
	"26_04"
	"26_03"
	"26_02"
	"26_05"
]


In [22]:
print(performance_all_site.schema)
print(performance_all_site['_PST.Date'].unique())

Schema({'Agent People Id': String, 'Business Segment Name': String, 'Partner Name': String, 'Customer Loyalty Status': String, 'Response Count': String, 'Response Time': String, 'Latest VA Product': String, 'Language': String, 'Latest VA Intent': String, 'Conversation Id': String, 'Agent Queue Group Name': String, 'Joined Time': Datetime(time_unit='us', time_zone=None), '_PST.Interval': String, 'Agent Email ID': String, 'Left Time': String, 'Handle (Count)': Int64, 'Handle Time (Sum)': Float64, 'Hold Time (Sum)': Float64, 'Talk Time (Sum)': Float64, 'Join Time (VNT)': Datetime(time_unit='us', time_zone=None), '_VNT.Interval': String, '_VNT.Period': String, 'Wrap Up Time (Sum)': Float64, 'Agent Business Location': String, 'CCR72': Float64, '_PST.Date': Date, '_PST.Month': String, '_PST.Year': Int32, '_aob': Int32, 'LOB': String, '_conver_unique': String, '_nps_type': String, '_promoter': Float64, '_detractor': Float64, '_neutral': Float64, '_survey': Float64, '_ir': Int32, '_ae': Int64,